<a href="https://colab.research.google.com/github/Alezgo-ui/Ecommerce_datacleaning_dashboard/blob/main/data_cleaning_eda_ecommerce_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarme con la estructura de los datasets del negocio antes de analizarlos.

**Procedimiento:**

- Importar las librerías necesarias
- Cargar los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guardar los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explorar cada dataset.

---

In [ ]:
# importar librerías
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# cargar archivos
orders = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv')
catalog = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv')
marketing = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv')

**Explorar datasets**

---

*PIPELINE DE REVISIÓN INICIAL DE DATASETS (EDA - Exploratory Data Analysis)*

---

*OBJETIVO*


Antes de modelar, limpiar o cruzar datasets, necesitamos entender:
  1. ¿Qué forma tiene el dataset? (filas, columnas, tipos)
  2. ¿Qué tan "sucio" está? (nulos, duplicados, inconsistencias)
  3. ¿Qué contiene cada variable? (rangos, categorías, distribución)
  4. ¿Es viable en memoria y está bien tipado?

Este pipeline automatiza esas preguntas para UNO o VARIOS datasets a la vez,
generando un reporte consistente y comparable entre ellos. Esto es clave
cuando se trabaja con varias fuentes (orden, catalogo, marketing, etc.) y
se necesita detectar rápido cuál tiene problemas antes de integrarlas.

In [ ]:
# ------------------------------------------------------------------------
# 1. FORMA Y ESTRUCTURA GENERAL
# ------------------------------------------------------------------------
def info_general(df: pd.DataFrame) -> dict:
    """
    Por qué: Es el primer chequeo obligado. Nos dice si el archivo se
    cargó bien (filas/columnas esperadas), cuánto pesa en memoria
    (relevante si luego haremos merges o lo subiremos a Power BI/BD)
    y un vistazo rápido a los tipos de dato que pandas infirió
    (a veces infiere mal: fechas como texto, IDs como float, etc.).
    """
    return {
        "filas": df.shape[0],
        "columnas": df.shape[1],
        "memoria_mb": round(df.memory_usage(deep=True).sum() / 1024**2, 2),
        "tipos_de_dato": df.dtypes.value_counts().to_dict(),
    }


# ------------------------------------------------------------------------
# 2. VALORES NULOS
# ------------------------------------------------------------------------
def resumen_nulos(df: pd.DataFrame) -> pd.DataFrame:
    """
    Por qué: Los nulos determinan qué estrategia de limpieza usar
    (imputar, eliminar, o dejar la columna fuera del análisis).
    Ordenamos de mayor a menor % para priorizar qué columnas revisar
    primero. Una columna con >50% de nulos casi siempre requiere
    una decisión explícita (¿se descarta? ¿se imputa? ¿el nulo
    significa algo, como "no aplica"?).
    """
    nulos = df.isnull().sum()
    porcentaje = (nulos / len(df) * 100).round(2)
    tabla = pd.DataFrame({"nulos": nulos, "porcentaje_%": porcentaje})
    return tabla[tabla["nulos"] > 0].sort_values("porcentaje_%", ascending=False)


# ------------------------------------------------------------------------
# 3. DUPLICADOS
# ------------------------------------------------------------------------
def resumen_duplicados(df: pd.DataFrame, subset: list = None) -> dict:
    """
    Por qué: Filas duplicadas inflan conteos, sumas y promedios sin que
    sea evidente. Es especialmente crítico antes de hacer un GROUP BY
    o una medida de participación (como en el dashboard de ventas):
    duplicados ahí distorsionan directamente el resultado.

    El parámetro `subset` permite chequear duplicados por una clave de
    negocio (ej. ["id_venta"]) en vez de la fila completa, que es más
    estricto y a veces esconde duplicados reales.
    """
    total_duplicados = df.duplicated(subset=subset).sum()
    resultado = {
        "duplicados_totales": int(total_duplicados),
        "porcentaje_%": round(total_duplicados / len(df) * 100, 2),
        "subset_evaluado": subset if subset else "fila completa",
    }

    # Imprime cada valor en su propia línea (en vez de solo regresar
    # el diccionario, que se ve pegado si se imprime tal cual).
    for clave, valor in resultado.items():
        print(f"{clave}: {valor}")

    return resultado


# ------------------------------------------------------------------------
# 4. RESUMEN POR COLUMNA (tipo, cardinalidad, ejemplos)
# ------------------------------------------------------------------------
def resumen_columnas(df: pd.DataFrame) -> pd.DataFrame:
    """
    Por qué: Antes de decidir el tratamiento de cada columna necesitamos
    saber:
      - dtype: ¿pandas la interpretó como el analista esperaba?
      - n_unicos: ¿es una variable categórica (pocos valores) o
        continua/ID (muchos valores únicos)? Esto define si conviene
        convertirla a `category` (ahorra memoria) o tratarla como
        numérica.
      - ejemplo: un vistazo rápido a valores reales, útil para detectar
        formatos inconsistentes (ej. "2024-01-01" vs "01/01/2024").
    """
    filas = []
    for col in df.columns:
        serie = df[col]
        filas.append({
            "columna": col,
            "dtype": str(serie.dtype),
            "n_unicos": serie.nunique(dropna=True),
            "%_unicos": round(serie.nunique(dropna=True) / len(df) * 100, 2),
            "ejemplo": serie.dropna().iloc[0] if serie.dropna().shape[0] > 0 else None,
        })
    return pd.DataFrame(filas)


# ------------------------------------------------------------------------
# 5. ESTADÍSTICAS DESCRIPTIVAS (numéricas y categóricas por separado)
# ------------------------------------------------------------------------
def estadisticas_numericas(df: pd.DataFrame) -> pd.DataFrame:
    """
    Por qué: Detecta outliers evidentes (min/max fuera de rango lógico,
    ej. una edad de -5 o un precio de 0), y da una primera idea de la
    distribución (media vs mediana muy distintas = asimetría/outliers).
    """
    numericas = df.select_dtypes(include=np.number)
    if numericas.empty:
        return pd.DataFrame()
    return numericas.describe().T.round(2)


def estadisticas_categoricas(df: pd.DataFrame, top_n: int = 5) -> dict:
    """
    Por qué: Para columnas de texto/categoría, describe() no es útil.
    Aquí mostramos las categorías más frecuentes, lo que ayuda a
    detectar inconsistencias de captura (ej. "CDMX", "Cdmx", "cdmx"
    tratadas como categorías distintas cuando deberían ser una sola).
    """
    categoricas = df.select_dtypes(include=["object", "category"])
    resultado = {}
    for col in categoricas.columns:
        resultado[col] = df[col].value_counts(dropna=False).head(top_n)

    # Imprime cada columna con su propio encabezado, separada del resto
    # (en vez de regresar solo el diccionario, que se ve desordenado si se imprime tal cual).
    for columna, conteo in resultado.items():
        print(f"\n--- {columna} ---")
        print(conteo)

    return resultado


# ------------------------------------------------------------------------
# 6. VALORES ÚNICOS DE UNA COLUMNA ESPECÍFICA
# ------------------------------------------------------------------------
def valores_unicos(df: pd.DataFrame, columna: str, umbral_cardinalidad: float = 50.0,
                    minimo_filas_para_bloqueo: int = 30):
    """
    Por qué: Permite explorar a detalle una sola columna categórica
    (ej. "pais") sin correr todas las columnas del dataset a la vez.

    Protección de cardinalidad: si la columna es una llave única o casi
    única (ej. id_pedido, con ~100% de valores distintos), imprimir
    todos sus valores no aporta nada y satura la salida. En ese caso,
    la función lo detecta y solo informa la cardinalidad, sin listar
    los valores.

    Parámetros
    ----------
    df : pd.DataFrame
    columna : str
        Nombre de la columna a explorar.
    umbral_cardinalidad : float
        % de valores únicos a partir del cual se considera que la
        columna es una llave (no categórica) y se bloquea el listado
        completo. Por defecto 50%.
   minimo_filas_para_bloqueo : int
        Número mínimo de filas que debe tener el DataFrame para que el
        bloqueo por cardinalidad aplique. Por defecto 30.
    """
    if columna not in df.columns:
        print(f"La columna '{columna}' no existe en el DataFrame.")
        return None

    porcentaje_unicos = round(df[columna].nunique(dropna=True) / len(df) * 100, 2)

    if len(df) >= minimo_filas_para_bloqueo and porcentaje_unicos > umbral_cardinalidad:
        print(f"'{columna}' tiene {porcentaje_unicos}% de valores únicos "
              f"(> {umbral_cardinalidad}%). Parece una llave/ID, no una "
              f"categoría — no se listan los valores para evitar saturar "
              f"la salida. Usa resumen_columnas(df) para más detalle.")
        return None

    conteo = df[columna].value_counts(dropna=False)
    print(f"\n--- {columna} ({porcentaje_unicos}% únicos) ---")
    print(conteo)
    return conteo


# ------------------------------------------------------------------------
# 7. PIPELINE PRINCIPAL: orquesta todo lo anterior para 1 o N datasets
# ------------------------------------------------------------------------
def revisar_dataset(df: pd.DataFrame, nombre: str = "dataset") -> dict:
    """
    Corre toda la batería de revisión sobre UN dataset y devuelve
    un diccionario con cada pieza del reporte, más un print en consola
    con el resumen ejecutivo para revisión rápida en el notebook.
    """
    reporte = {
        "info_general": info_general(df),
        "resumen": resumen_columnas(df),
        "nulos": resumen_nulos(df),
        "duplicados": resumen_duplicados(df),
        "estadisticas_numericas": estadisticas_numericas(df),
        "estadisticas_categoricas": estadisticas_categoricas(df),
    }

    print(f"\n{'='*70}\nREVISIÓN INICIAL: {nombre}\n{'='*70}")
    print(f"Filas: {reporte['info_general']['filas']:,} | "
          f"Columnas: {reporte['info_general']['columnas']} | "
          f"Memoria: {reporte['info_general']['memoria_mb']} MB")
    if not reporte["nulos"].empty:
        print(f"Columnas con nulos: {len(reporte['nulos'])}")
        print(reporte["nulos"].head(10))
    else:
        print("Sin valores nulos detectados.")

    return reporte


def revisar_datasets(datasets: dict) -> dict:
    """
    Punto de entrada para revisar VARIOS datasets a la vez.

    Por qué: cuando trabajas con múltiples fuentes (ej. ventas,
    clientes, inventario) quieres un reporte comparable y consistente
    entre todas, sin repetir el análisis a mano para cada una. Esto
    también ayuda a decidir el orden de limpieza (empezar por la más
    "sucia") antes de integrarlas o cruzarlas.

    Parámetros
    ----------
    datasets : dict[str, pd.DataFrame]
        Ej. {"ventas": df_ventas, "clientes": df_clientes}

    Retorna
    -------
    dict[str, dict]
        Un reporte completo por cada dataset (ver `revisar_dataset`).
    """
    reportes = {}
    for nombre, df in datasets.items():
        reportes[nombre] = revisar_dataset(df, nombre)
    return reportes

In [ ]:
datasets = {"orders": orders, "catalog": catalog, "marketing": marketing}

for nombre, df in datasets.items():
    print(f"\n{'='*70}\n{nombre.upper()}\n{'='*70}")

    print("\n--- Revisión inicial ---")
    print(info_general(df))
    print("\n")
    print(df.info())


ORDERS

--- Revisión inicial ---
{'filas': 25100, 'columnas': 12, 'memoria_mb': 13.45, 'tipos_de_dato': {dtype('O'): 8, dtype('float64'): 4}}


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), object(8)
memory usage: 2.3+ MB
None

CA

---

### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas

---

#### Revision y calidad de datos en el df "orders"

In [ ]:
print("--- Tipos de dato y cardinalidad ---")
display(resumen_columnas(orders))

print("\n--- Valores nulos ---")
display(resumen_nulos(orders))

print("\n--- Duplicados ---")
resumen_duplicados(orders)

print("\n--- Estadísticas numéricas ---")
display(estadisticas_numericas(orders))

print("\n--- Analisis categorico ---")
# Lista de columnas categóricas a revisar (excluye id_pedido e id_usuario,
# que son identificadores, no categorías).
cols_categoricas = ["pais", "dispositivo", "fuente_referencia",
                     "nombre_producto", "categoria_producto"]

# Recorre cada columna y muestra sus valores únicos con conteo,
# para detectar inconsistencias de capitalización/espacios como las que encontramos en pais (ej. "Mexico" vs "mexico").
for col in cols_categoricas:
    _ = valores_unicos(orders, col)# el "_" evita que Jupyter la muestre otra vez


--- Tipos de dato y cardinalidad ---


,columna,dtype,n_unicos,%_unicos,ejemplo
0,id_pedido,object,25000,99.60,order_0
1,id_usuario,object,7642,30.45,user_6993
2,fecha_hora_pedido,object,181,0.72,2025-05-22
3,pais,object,6,0.02,Argentina
4,dispositivo,object,2,0.01,desktop
5,fuente_referencia,object,3,0.01,organic
6,nombre_producto,object,7,0.03,Jacket-Winter-M
7,categoria_producto,object,3,0.01,Moda
8,cantidad,float64,6,0.02,2.0
9,precio_unitario,float64,19543,77.86,332.69



--- Valores nulos ---


,nulos,porcentaje_%
pais,300,1.20
categoria_producto,80,0.32
cantidad,50,0.20
precio_unitario,50,0.20
monto_descuento,50,0.20
fuente_referencia,30,0.12
nombre_producto,30,0.12
dispositivo,20,0.08



--- Duplicados ---
duplicados_totales: 100
porcentaje_%: 0.4
subset_evaluado: fila completa

--- Estadísticas numéricas ---


,count,mean,std,min,25%,50%,75%,max
cantidad,25050.0,7.09,296.28,-2.00,1.00,2.00,2.00,20000.00
precio_unitario,25050.0,259.31,138.73,20.03,138.38,258.72,380.33,499.96
monto_descuento,25050.0,4.50,5.22,0.00,0.00,0.00,10.00,15.00
monto_total,25100.0,2072.68,98949.95,-492.65,180.51,341.75,518.58,8840200.00



--- Analisis categorico ---

--- pais (0.02% únicos) ---
Colombia     7520
Mexico       7502
Argentina    7291
mexico        865
colombia      823
argentina     799
NaN           300
Name: pais, dtype: int64

--- dispositivo (0.01% únicos) ---
desktop    12759
mobile     12321
NaN           20
Name: dispositivo, dtype: int64

--- fuente_referencia (0.01% únicos) ---
social         8428
organic        8329
paid_search    8313
NaN              30
Name: fuente_referencia, dtype: int64

--- nombre_producto (0.03% únicos) ---
Vacuum-Pro-Black        4199
Blender-XL-Red          4195
Jacket-Winter-M         4192
Sneakers-Urban-42       4160
Laptop-Gaming-16GB      2794
Tablet-Standard-64GB    2780
Phone-Pro-128GB         2750
NaN                       30
Name: nombre_producto, dtype: int64

--- categoria_producto (0.01% únicos) ---
Hogar          8385
Moda           8323
Electronica    8312
NaN              80
Name: categoria_producto, dtype: int64


**Copia inicial**

In [ ]:
# Copia explícita antes de modificar, para no alterar orders (el
# original) y evitar el SettingWithCopyWarning en pasos posteriores.
orders_clean = orders.copy()
filas_iniciales = orders_clean.shape[0]

**Conversión de tipos**

In [ ]:
# fecha_hora_pedido: de texto (object) a datetime, para poder filtrar
# por fecha o cruzar con marketing más adelante.
orders_clean["fecha_hora_pedido"] = pd.to_datetime(orders_clean["fecha_hora_pedido"], errors="coerce")

# Columnas categóricas de baja cardinalidad → tipo category (ahorra memoria y deja explícito que son categorías, no texto libre).
# id_pedido e id_usuario quedan fuera por ser identificadores.
cols_categoricas = ["pais", "dispositivo", "fuente_referencia",
                     "nombre_producto", "categoria_producto"]
for col in cols_categoricas:
    orders_clean[col] = orders_clean[col].astype("category")

resumen_columnas(orders_clean)

,columna,dtype,n_unicos,%_unicos,ejemplo
0,id_pedido,object,25000,99.60,order_0
1,id_usuario,object,7642,30.45,user_6993
2,fecha_hora_pedido,datetime64[ns],181,0.72,2025-05-22 00:00:00
3,pais,category,6,0.02,Argentina
4,dispositivo,category,2,0.01,desktop
5,fuente_referencia,category,3,0.01,organic
6,nombre_producto,category,7,0.03,Jacket-Winter-M
7,categoria_producto,category,3,0.01,Moda
8,cantidad,float64,6,0.02,2.0
9,precio_unitario,float64,19543,77.86,332.69


**Unificar pais**

In [ ]:
# Unifica capitalización: "Mexico"/"mexico" y variantes similares se
# contaban como categorías distintas. Se pasa a texto, se limpia, y se regresa a category.
orders_clean["pais"] = orders_clean["pais"].astype(str).str.strip().str.title().astype("category")
orders_clean["pais"].value_counts(dropna=False)

Mexico       8367
Colombia     8343
Argentina    8090
NaN           300
Name: pais, dtype: int64

**Duplicados**

In [ ]:
# Compara duplicados por id_pedido (llave de negocio) contra los
# 100 duplicados por fila completa que ya conocíamos.
resumen_duplicados(orders_clean, subset=["id_pedido"])

duplicados_totales: 100
porcentaje_%: 0.4
subset_evaluado: ['id_pedido']


{'duplicados_totales': 100,
 'porcentaje_%': 0.4,
 'subset_evaluado': ['id_pedido']}

In [ ]:
# Elimina duplicados por id_pedido (llave real de negocio), no por
# fila completa. Se confirmó antes que ambos conteos coinciden (100),
# así que es seguro eliminar directo.
filas_antes_dup = orders_clean.shape[0]
orders_clean = orders_clean.drop_duplicates(subset=["id_pedido"], keep="first")
print(f"Duplicados eliminados: {filas_antes_dup - orders_clean.shape[0]}")
print("Filas después de eliminar duplicados:", orders_clean.shape[0])

Duplicados eliminados: 100
Filas después de eliminar duplicados: 25000


**Valores negativos (posibles devoluciones)**

In [ ]:
# Cuantifica cuántas filas tienen cantidad <= 0 o monto_total negativo
print("Filas con cantidad <= 0:", (orders["cantidad"] <= 0).sum())
print("Filas con monto_total negativo:", (orders["monto_total"] < 0).sum())

# Vistazo a esas filas para entender si son error de captura o
# reembolsos/devoluciones legítimas
orders[(orders["cantidad"] <= 0) | (orders["monto_total"] < 0)].head(10)

Filas con cantidad <= 0: 4
Filas con monto_total negativo: 4


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
266,order_266,user_7011,2025-03-13,NaN,desktop,paid_search,Phone-Pro-128GB,Electronica,-2.0,101.31,10.0,-192.62
267,order_267,user_1087,2025-05-07,NaN,desktop,social,Phone-Pro-128GB,Electronica,-1.0,43.50,5.0,-38.50
268,order_268,user_84,2025-02-19,NaN,desktop,organic,Phone-Pro-128GB,Electronica,-1.0,497.65,5.0,-492.65
269,order_269,user_3323,2025-05-25,NaN,desktop,paid_search,Phone-Pro-128GB,Electronica,-1.0,423.53,0.0,-423.53


In [ ]:
# Marca las filas con cantidad negativa como devoluciones en vez de
# eliminarlas o "corregirlas" — son datos reales (reembolsos), no
# errores de captura, confirmado al validar la fórmula de monto_total.
orders_clean["es_devolucion"] = orders_clean["cantidad"] < 0
print(f"Devoluciones marcadas: {orders_clean['es_devolucion'].sum()}")

Devoluciones marcadas: 4


Los 4 registros muestran un patrón muy claro, y las cifras cuadran perfectamente — esto no es basura, es un patrón de negocio legítimo:
Esto tiene sentido de negocio: son devoluciones/reembolsos. Si un pedido original fue precio × cantidad - descuento, al devolverlo el reembolso es exactamente ese monto en negativo: precio × (-cantidad) + descuento. No es un error de captura — es el sistema registrando reembolsos como filas separadas con cantidad negativa.

Mi recomendación : no eliminar ni "corregir" estas filas — tratarlas como una categoría de negocio propia (devoluciones), no como error.

**Eliminar nulos numéricos**

In [ ]:
# Elimina pedidos donde cantidad, precio_unitario y monto_descuento
# están nulos simultáneamente (confirmado: siempre coinciden en las
# mismas filas) — sin estos datos no se puede calcular monto_total
# de forma confiable.
filas_antes_nulos = orders_clean.shape[0]
orders_clean = orders_clean.dropna(subset=["cantidad", "precio_unitario", "monto_descuento"])
print(f"Filas eliminadas por nulos numéricos núcleo: {filas_antes_nulos - orders_clean.shape[0]}")

Filas eliminadas por nulos numéricos núcleo: 50


In [ ]:
# Revisar si los nulos de estas 3 columnas coinciden en las mismas filas
nulos_numericos = orders_clean[["cantidad", "precio_unitario", "monto_descuento"]].isnull().sum(axis=1)
nulos_numericos.value_counts()

0    24950
dtype: int64

Confirmado: 50 filas tienen los 3 valores nulos a la vez (0 nulos en 24,950 filas y 3 nulos exactamente en 50 filas) — no hay casos mixtos de solo 1 o 2 nulos. Esto confirma que es un mismo grupo de pedidos con la información numérica núcleo completamente ausente, así que es seguro y limpio eliminarlas.

In [ ]:
# Eliminar los 50 pedidos con cantidad, precio_unitario y monto_descuento
# nulos a la vez — sin estos datos no se puede calcular monto_total
# ni analizar el pedido de forma confiable.
filas_antes = orders_clean.shape[0]
orders_clean = orders_clean.dropna(subset=["cantidad", "precio_unitario", "monto_descuento"])
print(f"Filas eliminadas: {filas_antes - orders_clean.shape[0]}")
print(f"Filas restantes: {orders_clean.shape[0]}")

Filas eliminadas: 0
Filas restantes: 24950


**Imputar nulos categóricos de bajo volumen**

In [ ]:
# Para columnas con pocos nulos (≤1.2%), imputa como "Desconocido"
# en vez de eliminar filas con datos válidos en el resto de columnas.
for col in ["pais", "fuente_referencia", "nombre_producto", "dispositivo"]:
    orders_clean[col] = orders_clean[col].cat.add_categories("Desconocido").fillna("Desconocido")

**Imputar categoria_producto cruzando con catalog**

In [ ]:
mapa_categoria = (
    catalog.set_index("nombre_producto")["categoria_producto"]
    .str.strip()
    .str.replace("ó", "o", regex=False)
)

valores_a_imputar = orders_clean["nombre_producto"].map(mapa_categoria)
categorias_nuevas = set(valores_a_imputar.dropna().unique()) - set(orders_clean["categoria_producto"].cat.categories)
orders_clean["categoria_producto"] = orders_clean["categoria_producto"].cat.add_categories(categorias_nuevas)
orders_clean["categoria_producto"] = orders_clean["categoria_producto"].fillna(valores_a_imputar)
orders_clean["categoria_producto"] = orders_clean["categoria_producto"].cat.add_categories("Desconocido").fillna("Desconocido")

**Analisis de outlier extremo**

In [ ]:
# Aísla las filas con los montos más extremos (percentil 99.9) para
# revisar si son error de captura o pedidos reales de alto valor.
orders_clean[orders_clean["monto_total"] > orders_clean["monto_total"].quantile(0.999)].sort_values("monto_total", ascending=False)

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total,es_devolucion
3722,order_3722,user_4723,2025-05-09,Argentina,mobile,paid_search,Laptop-Gaming-16GB,Electronica,20000.0,442.01,0.0,8840200.00,False
3668,order_3668,user_7270,2025-06-24,Mexico,mobile,paid_search,Laptop-Gaming-16GB,Electronica,20000.0,348.31,0.0,6966200.00,False
3656,order_3656,user_884,2025-01-01,Argentina,mobile,organic,Laptop-Gaming-16GB,Electronica,20000.0,297.66,0.0,5953200.00,False
3726,order_3726,user_2536,2025-02-12,Colombia,desktop,social,Laptop-Gaming-16GB,Electronica,20000.0,290.85,0.0,5817000.00,False
3586,order_3586,user_3380,2025-02-03,Mexico,mobile,paid_search,Laptop-Gaming-16GB,Electronica,10000.0,490.35,0.0,4903500.00,False
3748,order_3748,user_7096,2025-02-23,Mexico,desktop,organic,Laptop-Gaming-16GB,Electronica,10000.0,336.93,0.0,3369300.00,False
3522,order_3522,user_3575,2025-03-29,Argentina,desktop,social,Laptop-Gaming-16GB,Electronica,10000.0,280.55,0.0,2805500.00,False
3643,order_3643,user_4440,2025-01-07,Colombia,desktop,social,Laptop-Gaming-16GB,Electronica,10000.0,238.15,0.0,2381500.00,False
3689,order_3689,user_6566,2025-06-16,Mexico,desktop,paid_search,Laptop-Gaming-16GB,Electronica,10000.0,87.69,0.0,876900.00,False
3521,order_3521,user_5812,2025-02-03,Mexico,mobile,paid_search,Laptop-Gaming-16GB,Electronica,10000.0,43.14,0.0,431400.00,False


In [ ]:
# Aísla todos los casos con cantidad anormalmente alta (por encima de 100 unidades — el resto del dataset opera en un rango de 1-10).
sospechosos = orders_clean[orders_clean["cantidad"] > 100]
print(f"Filas con cantidad > 100: {sospechosos.shape[0]}")
print(f"% del total: {sospechosos.shape[0] / orders_clean.shape[0] * 100:.2f}%")

sospechosos[["nombre_producto", "cantidad", "precio_unitario", "monto_total"]].sort_values("cantidad", ascending=False)

Filas con cantidad > 100: 10
% del total: 0.04%


,nombre_producto,cantidad,precio_unitario,monto_total
3656,Laptop-Gaming-16GB,20000.0,297.66,5953200.0
3668,Laptop-Gaming-16GB,20000.0,348.31,6966200.0
3722,Laptop-Gaming-16GB,20000.0,442.01,8840200.0
3726,Laptop-Gaming-16GB,20000.0,290.85,5817000.0
3521,Laptop-Gaming-16GB,10000.0,43.14,431400.0
3522,Laptop-Gaming-16GB,10000.0,280.55,2805500.0
3586,Laptop-Gaming-16GB,10000.0,490.35,4903500.0
3643,Laptop-Gaming-16GB,10000.0,238.15,2381500.0
3689,Laptop-Gaming-16GB,10000.0,87.69,876900.0
3748,Laptop-Gaming-16GB,10000.0,336.93,3369300.0


Confirmado: exactamente 10 filas (0.04% del dataset), y todas son Laptop-Gaming-16GB con cantidad de 10,000 o 20,000 — sin excepción. Esto descarta que sea un problema disperso; es un error de captura aislado y específico a ese producto, probablemente en el proceso de carga original de los datos.

Con el alcance confirmado, procedo a la eliminación:

In [ ]:
# Elimina las 10 filas con cantidad claramente errónea (>100 unidades,
# confirmado: todas son Laptop-Gaming-16GB con cantidad de 10,000-20,000).
# No hay forma de recuperar el valor real de cantidad, y dejarlas distorsionaría cualquier suma o promedio de ventas.
filas_antes_outliers = orders_clean.shape[0]
orders_clean = orders_clean[orders_clean["cantidad"] <= 100]
print(f"Filas eliminadas por cantidad anómala: {filas_antes_outliers - orders_clean.shape[0]}")
print(f"Filas restantes: {orders_clean.shape[0]}")

# Confirma que las estadísticas ya no muestran el outlier extremo
estadisticas_numericas(orders_clean)

Filas eliminadas por cantidad anómala: 10
Filas restantes: 24940


,count,mean,std,min,25%,50%,75%,max
cantidad,24940.0,1.50,0.50,-2.00,1.00,2.00,2.00,2.00
precio_unitario,24940.0,259.36,138.69,20.03,138.46,258.77,380.40,499.96
monto_descuento,24940.0,4.50,5.22,0.00,0.00,0.00,10.00,15.00
monto_total,24940.0,385.77,255.82,-492.65,180.59,341.36,517.63,999.89


**Confirmación final**

In [ ]:
# Verifica que orders_clean quedó sin nulos y con el conteo de filas
# esperado, antes de darlo por terminado.
print(f"Filas iniciales: {filas_iniciales} | Filas finales: {orders_clean.shape[0]}")

print("--- Tipos de dato y cardinalidad ---")
display(resumen_columnas(orders_clean))

print("\n--- Valores nulos ---")
display(resumen_nulos(orders_clean))

print("\n--- Duplicados ---")
resumen_duplicados(orders_clean)

print("\n--- Estadísticas numéricas ---")
display(estadisticas_numericas(orders_clean))

print("\n--- Analisis categorico ---")
# Lista de columnas categóricas a revisar (excluye id_pedido e id_usuario,
# que son identificadores, no categorías).
cols_categoricas = ["pais", "dispositivo", "fuente_referencia",
                     "nombre_producto", "categoria_producto"]

# Recorre cada columna y muestra sus valores únicos con conteo,
# para detectar inconsistencias de capitalización/espacios como
# las que encontramos en pais (ej. "Mexico" vs "mexico").
for col in cols_categoricas:
    _ = valores_unicos(orders_clean, col)# el "_" evita que Jupyter la muestre otra vez

Filas iniciales: 25100 | Filas finales: 24940
--- Tipos de dato y cardinalidad ---


,columna,dtype,n_unicos,%_unicos,ejemplo
0,id_pedido,object,24940,100.00,order_0
1,id_usuario,object,7640,30.63,user_6993
2,fecha_hora_pedido,datetime64[ns],181,0.73,2025-05-22 00:00:00
3,pais,category,4,0.02,Argentina
4,dispositivo,category,3,0.01,desktop
5,fuente_referencia,category,4,0.02,organic
6,nombre_producto,category,8,0.03,Jacket-Winter-M
7,categoria_producto,category,4,0.02,Moda
8,cantidad,float64,4,0.02,2.0
9,precio_unitario,float64,19535,78.33,332.69



--- Valores nulos ---


,nulos,porcentaje_%



--- Duplicados ---
duplicados_totales: 0
porcentaje_%: 0.0
subset_evaluado: fila completa

--- Estadísticas numéricas ---


,count,mean,std,min,25%,50%,75%,max
cantidad,24940.0,1.50,0.50,-2.00,1.00,2.00,2.00,2.00
precio_unitario,24940.0,259.36,138.69,20.03,138.46,258.77,380.40,499.96
monto_descuento,24940.0,4.50,5.22,0.00,0.00,0.00,10.00,15.00
monto_total,24940.0,385.77,255.82,-492.65,180.59,341.36,517.63,999.89



--- Analisis categorico ---

--- pais (0.02% únicos) ---
Mexico         8322
Colombia       8287
Argentina      8031
Desconocido     300
Name: pais, dtype: int64

--- dispositivo (0.01% únicos) ---
desktop        12680
mobile         12240
Desconocido       20
Name: dispositivo, dtype: int64

--- fuente_referencia (0.02% únicos) ---
social         8383
organic        8268
paid_search    8259
Desconocido      30
Name: fuente_referencia, dtype: int64

--- nombre_producto (0.03% únicos) ---
Blender-XL-Red          4176
Vacuum-Pro-Black        4170
Jacket-Winter-M         4166
Sneakers-Urban-42       4129
Laptop-Gaming-16GB      2768
Tablet-Standard-64GB    2764
Phone-Pro-128GB         2737
Desconocido               30
Name: nombre_producto, dtype: int64

--- categoria_producto (0.02% únicos) ---
Hogar          8346
Moda           8295
Electronica    8269
Desconocido      30
Name: categoria_producto, dtype: int64


#### Revision y calidad de datos en el df "catalog"

In [ ]:
print("--- Tipos de dato y cardinalidad ---")
display(resumen_columnas(catalog))

print("\n--- Valores nulos ---")
display(resumen_nulos(catalog))

print("\n--- Duplicados ---")
resumen_duplicados(catalog)

print("\n--- Estadísticas numéricas ---")
display(estadisticas_numericas(catalog))

print("\n--- Analisis categorico ---")
# Lista de columnas categóricas a revisar (excluye id_pedido e id_usuario,
# que son identificadores, no categorías).
cols_categoricas = ["nombre_producto", "categoria_producto", "proveedor"]

# Recorre cada columna y muestra sus valores únicos con conteo,
# para detectar inconsistencias de capitalización/espacios como
# las que encontramos en pais (ej. "Mexico" vs "mexico").
for col in cols_categoricas:
    _ = valores_unicos(catalog, col)# el "_" evita que Jupyter la muestre otra vez

--- Tipos de dato y cardinalidad ---


,columna,dtype,n_unicos,%_unicos,ejemplo
0,nombre_producto,object,7,100.00,Laptop-Gaming-16GB
1,categoria_producto,object,3,42.86,Electrónica
2,costo_unitario,float64,7,100.00,280.68
3,proveedor,object,7,100.00,"Fuller, Pena and Myers"



--- Valores nulos ---


,nulos,porcentaje_%



--- Duplicados ---
duplicados_totales: 0
porcentaje_%: 0.0
subset_evaluado: fila completa

--- Estadísticas numéricas ---


,count,mean,std,min,25%,50%,75%,max
costo_unitario,7.0,102.25,111.01,10.12,16.9,25.21,182.98,280.68



--- Analisis categorico ---

--- nombre_producto (100.0% únicos) ---
Jacket-Winter-M         1
Tablet-Standard-64GB    1
Blender-XL-Red          1
Phone-Pro-128GB         1
Laptop-Gaming-16GB      1
Sneakers-Urban-42       1
Vacuum-Pro-Black        1
Name: nombre_producto, dtype: int64

--- categoria_producto (42.86% únicos) ---
Electrónica    3
Hogar          2
Moda           2
Name: categoria_producto, dtype: int64

--- proveedor (100.0% únicos) ---
Greene-Smith               1
Long-Reid                  1
Fuller, Pena and Myers     1
Bowers LLC                 1
King Ltd                   1
Rivera, Carr and Finley    1
Mcmillan-Rhodes            1
Name: proveedor, dtype: int64


**Copia inicial**

In [ ]:
# Copia explícita antes de modificar, para no alterar catalog (el original).
catalog_clean = catalog.copy()

**Corregir el acento en categoria_producto**

In [ ]:
# Quita acentos/espacios extra en categoria_producto para que coincida
# exactamente con el formato usado en orders_clean (sin acentos).
# Esto es lo que generaba la categoría fantasma "Electrónica" al cruzar.
catalog_clean["categoria_producto"] = (
    catalog_clean["categoria_producto"].str.strip().str.replace("ó", "o", regex=False)
)

catalog_clean["categoria_producto"].value_counts(dropna=False)

Electronica    3
Hogar          2
Moda           2
Name: categoria_producto, dtype: int64

**Conversión de tipos**

In [ ]:
catalog_clean["categoria_producto"] = catalog_clean["categoria_producto"].astype("category")
catalog_clean["nombre_producto"] = catalog_clean["nombre_producto"].astype("category")
resumen_columnas(catalog_clean)

,columna,dtype,n_unicos,%_unicos,ejemplo
0,nombre_producto,category,7,100.00,Laptop-Gaming-16GB
1,categoria_producto,category,3,42.86,Electronica
2,costo_unitario,float64,7,100.00,280.68
3,proveedor,object,7,100.00,"Fuller, Pena and Myers"


**Confirmación final**

In [ ]:
print("--- Tipos de dato y cardinalidad ---")
display(resumen_columnas(catalog_clean))

print("\n--- Valores nulos ---")
display(resumen_nulos(catalog_clean))

print("\n--- Duplicados ---")
resumen_duplicados(catalog_clean)

print("\n--- Estadísticas numéricas ---")
display(estadisticas_numericas(catalog_clean))

print("\n--- Analisis categorico ---")
# Lista de columnas categóricas a revisar (excluye id_pedido e id_usuario,
# que son identificadores, no categorías).
cols_categoricas = ["nombre_producto", "categoria_producto", "proveedor"]

# Recorre cada columna y muestra sus valores únicos con conteo,
# para detectar inconsistencias de capitalización/espacios como
# las que encontramos en pais (ej. "Mexico" vs "mexico").
for col in cols_categoricas:
    _ = valores_unicos(catalog_clean, col)# el "_" evita que Jupyter la muestre otra vez

--- Tipos de dato y cardinalidad ---


,columna,dtype,n_unicos,%_unicos,ejemplo
0,nombre_producto,category,7,100.00,Laptop-Gaming-16GB
1,categoria_producto,category,3,42.86,Electronica
2,costo_unitario,float64,7,100.00,280.68
3,proveedor,object,7,100.00,"Fuller, Pena and Myers"



--- Valores nulos ---


,nulos,porcentaje_%



--- Duplicados ---
duplicados_totales: 0
porcentaje_%: 0.0
subset_evaluado: fila completa

--- Estadísticas numéricas ---


,count,mean,std,min,25%,50%,75%,max
costo_unitario,7.0,102.25,111.01,10.12,16.9,25.21,182.98,280.68



--- Analisis categorico ---

--- nombre_producto (100.0% únicos) ---
Blender-XL-Red          1
Jacket-Winter-M         1
Laptop-Gaming-16GB      1
Phone-Pro-128GB         1
Sneakers-Urban-42       1
Tablet-Standard-64GB    1
Vacuum-Pro-Black        1
Name: nombre_producto, dtype: int64

--- categoria_producto (42.86% únicos) ---
Electronica    3
Hogar          2
Moda           2
Name: categoria_producto, dtype: int64

--- proveedor (100.0% únicos) ---
Greene-Smith               1
Long-Reid                  1
Fuller, Pena and Myers     1
Bowers LLC                 1
King Ltd                   1
Rivera, Carr and Finley    1
Mcmillan-Rhodes            1
Name: proveedor, dtype: int64


#### Revision y calidad de datos en el df "marketing"

In [ ]:
print("--- Tipos de dato y cardinalidad ---")
display(resumen_columnas(marketing))

print("\n--- Valores nulos ---")
display(resumen_nulos(marketing))

print("\n--- Duplicados ---")
resumen_duplicados(marketing)

print("\n--- Estadísticas numéricas ---")
display(estadisticas_numericas(marketing))

print("\n--- Analisis categorico ---")
# Lista de columnas categóricas a revisar (excluye id_pedido e id_usuario,
# que son identificadores, no categorías).
cols_categoricas = ["pais", "id_campaña", "canal"]

# Recorre cada columna y muestra sus valores únicos con conteo,
# para detectar inconsistencias de capitalización/espacios como
# las que encontramos en pais (ej. "Mexico" vs "mexico").
for col in cols_categoricas:
    _ = valores_unicos(marketing, col)# el "_" evita que Jupyter la muestre otra vez

--- Tipos de dato y cardinalidad ---


,columna,dtype,n_unicos,%_unicos,ejemplo
0,fecha,object,180,11.11,2025-01-01
1,pais,object,3,0.19,Mexico
2,id_campaña,object,9,0.56,organic_Mexico
3,canal,object,3,0.19,organic
4,gasto,float64,1612,99.51,2446.25



--- Valores nulos ---


,nulos,porcentaje_%
canal,101,6.23



--- Duplicados ---
duplicados_totales: 0
porcentaje_%: 0.0
subset_evaluado: fila completa

--- Estadísticas numéricas ---


,count,mean,std,min,25%,50%,75%,max
gasto,1620.0,1772.74,734.43,501.11,1128.03,1782.42,2420.68,2999.36



--- Analisis categorico ---

--- pais (0.19% únicos) ---
Colombia     540
Argentina    540
Mexico       540
Name: pais, dtype: int64

--- id_campaña (0.56% únicos) ---
social_Argentina         180
paid_search_Mexico       180
organic_Mexico           180
paid_search_Argentina    180
paid_search_Colombia     180
social_Colombia          180
social_Mexico            180
organic_Colombia         180
organic_Argentina        180
Name: id_campaña, dtype: int64

--- canal (0.19% únicos) ---
paid_search    507
organic        506
social         506
NaN            101
Name: canal, dtype: int64


**Copia explícita antes de modificar**

In [ ]:
marketing_clean = marketing.copy()

**Convertir tipos de dato**

In [ ]:
# Convierte fecha a datetime
marketing_clean["fecha"] = pd.to_datetime(marketing_clean["fecha"], errors="coerce")

**canal se puede imputar sin adivinar**

Como id_campaña sigue el patrón canal_Pais (ej. social_Colombia, search_Mexico) y no tiene ningún nulo, se puede recuperar el canal faltante extrayéndolo directo de id_campaña — no es una imputación estadística, es un dato que ya está ahí, solo hay que extraerlo:

In [ ]:
# Extrae el canal desde id_campaña (formato: canal_Pais) para las filas
# donde canal es nulo — es un dato recuperable, no una suposición.
canal_desde_id = marketing_clean["id_campaña"].str.rsplit("_", n=1).str[0]
marketing_clean["canal"] = marketing_clean["canal"].fillna(canal_desde_id)

print("Nulos restantes en canal:", marketing_clean["canal"].isnull().sum())

Nulos restantes en canal: 0


**Conversion de tipos categoricos**

In [ ]:
# Convierte pais, id_campaña y canal a category (baja cardinalidad,
# consistente con lo que hicimos en orders_clean y catalog_clean).
cols_categoricas_marketing = ["pais", "id_campaña", "canal"]
for col in cols_categoricas_marketing:
    marketing_clean[col] = marketing_clean[col].astype("category")

resumen_columnas(marketing_clean)

,columna,dtype,n_unicos,%_unicos,ejemplo
0,fecha,datetime64[ns],180,11.11,2025-01-01 00:00:00
1,pais,category,3,0.19,Mexico
2,id_campaña,category,9,0.56,organic_Mexico
3,canal,category,3,0.19,organic
4,gasto,float64,1612,99.51,2446.25


**Verificar outliers en gasto con IQR**

In [ ]:
# Calcula el rango intercuartílico (IQR) para detectar outliers en
# gasto, con el mismo criterio que usamos en orders_clean.
Q1 = marketing_clean["gasto"].quantile(0.25)
Q3 = marketing_clean["gasto"].quantile(0.75)
IQR = Q3 - Q1
limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

outliers_gasto = marketing_clean[(marketing_clean["gasto"] < limite_inferior) | (marketing_clean["gasto"] > limite_superior)]
print(f"Outliers en gasto: {outliers_gasto.shape[0]} ({outliers_gasto.shape[0]/len(marketing_clean)*100:.2f}%)")
print(f"Límite superior esperado: {limite_superior:.2f}")
outliers_gasto[["fecha", "pais", "canal", "gasto"]].sort_values("gasto", ascending=False).head(10)
if outliers_gasto.empty:
    print("Sin outliers detectados en gasto.")

Outliers en gasto: 0 (0.00%)
Límite superior esperado: 4359.67
Sin outliers detectados en gasto.


**Confirmacion final**

In [ ]:
print("--- Tipos de dato y cardinalidad ---")
display(resumen_columnas(marketing_clean))

print("\n--- Valores nulos ---")
display(resumen_nulos(marketing_clean))

print("\n--- Duplicados ---")
resumen_duplicados(marketing_clean)

print("\n--- Estadísticas numéricas ---")
display(estadisticas_numericas(marketing_clean))

print("\n--- Analisis categorico ---")
# Lista de columnas categóricas a revisar (excluye id_pedido e id_usuario,
# que son identificadores, no categorías).
cols_categoricas = ["pais", "id_campaña", "canal"]

# Recorre cada columna y muestra sus valores únicos con conteo,
# para detectar inconsistencias de capitalización/espacios como
# las que encontramos en pais (ej. "Mexico" vs "mexico").
for col in cols_categoricas:
    _ = valores_unicos(marketing_clean, col)# el "_" evita que Jupyter la muestre otra vez

--- Tipos de dato y cardinalidad ---


,columna,dtype,n_unicos,%_unicos,ejemplo
0,fecha,datetime64[ns],180,11.11,2025-01-01 00:00:00
1,pais,category,3,0.19,Mexico
2,id_campaña,category,9,0.56,organic_Mexico
3,canal,category,3,0.19,organic
4,gasto,float64,1612,99.51,2446.25



--- Valores nulos ---


,nulos,porcentaje_%



--- Duplicados ---
duplicados_totales: 0
porcentaje_%: 0.0
subset_evaluado: fila completa

--- Estadísticas numéricas ---


,count,mean,std,min,25%,50%,75%,max
gasto,1620.0,1772.74,734.43,501.11,1128.03,1782.42,2420.68,2999.36



--- Analisis categorico ---

--- pais (0.19% únicos) ---
Argentina    540
Colombia     540
Mexico       540
Name: pais, dtype: int64

--- id_campaña (0.56% únicos) ---
organic_Argentina        180
organic_Colombia         180
organic_Mexico           180
paid_search_Argentina    180
paid_search_Colombia     180
paid_search_Mexico       180
social_Argentina         180
social_Colombia          180
social_Mexico            180
Name: id_campaña, dtype: int64

--- canal (0.19% únicos) ---
organic        540
paid_search    540
social         540
Name: canal, dtype: int64


---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del analisis.

In [ ]:
# exportar datasets
orders_clean.to_csv('orders_clean.csv', index=False)
catalog_clean.to_csv('catalog_clean.csv', index=False)
marketing_clean.to_csv('marketing_clean.csv', index=False)

---

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)?
- ¿Cuál es el costo total?
- ¿Cuánto se ha invertido en marketing?
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden?
- ¿Cuál es la cantidad promedio de productos por orden?
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal?

**Cruce con catalog_clean**

In [ ]:
# Cruza orders_clean con catalog_clean para traer costo_unitario, necesario para calcular el costo total. "left" conserva todos los
# pedidos, incluso los que no tengan match (ej. nombre_producto = "Desconocido").
orders_full = orders_clean.merge(
    catalog_clean[["nombre_producto", "costo_unitario"]],
    on="nombre_producto",
    how="left"
)

print(f"Filas antes del cruce: {orders_clean.shape[0]}")
print(f"Filas después del cruce: {orders_full.shape[0]}")
print(f"Pedidos sin match de costo: {orders_full['costo_unitario'].isnull().sum()}")

Filas antes del cruce: 24940
Filas después del cruce: 24940
Pedidos sin match de costo: 30


**Calculos de KPI'S**

In [ ]:
# Calcula el costo total de cada pedido (costo_unitario x cantidad). Los 30 pedidos sin match (nombre_producto "Desconocido") quedarán
# con costo_total_pedido en NaN, y no se incluirán en la suma.
orders_full["costo_total_pedido"] = orders_full["costo_unitario"] * orders_full["cantidad"]

In [ ]:
# Calcula el costo total de cada pedido (costo_unitario x cantidad). Los 30 pedidos sin match (nombre_producto "Desconocido") quedarán
# con costo_total_pedido en NaN, y no se incluirán en la suma.
orders_full["costo_total_pedido"] = orders_full["costo_unitario"] * orders_full["cantidad"]

In [ ]:
# Ingreso total: suma de monto_total (ya neto, las devoluciones negativas se descuentan automáticamente).
ingreso_total = orders_full["monto_total"].sum()

# Costo total: suma de costo_total_pedido (los 30 pedidos "Desconocido" no aportan al costo, ya que no se puede calcular sin su categoría/costo).
costo_total = orders_full["costo_total_pedido"].sum()

# Inversión en marketing: total directo de marketing_clean.
inversion_marketing = marketing_clean["gasto"].sum()

# Profit = ingreso - costo de producto - inversión en marketing.
profit = ingreso_total - costo_total - inversion_marketing

**Rentabilidad del negocio**

In [ ]:
print(f"Ingreso total: ${ingreso_total:,.2f}")
print(f"Costo total: ${costo_total:,.2f}")
print(f"Inversión en marketing: ${inversion_marketing:,.2f}")
print(f"Profit: ${profit:,.2f}")
print(f"¿Rentable?: {'Sí' if profit > 0 else 'No'}")


Ingreso total: $9,621,134.26
Costo total: $3,828,818.41
Inversión en marketing: $2,871,843.53
Profit: $2,920,472.32
¿Rentable?: Sí


Una nota rápida de contexto, el negocio es rentable con un margen saludable — el profit representa cerca del 30% del ingreso total. Vale la pena mencionar que este profit no descuenta los 30 pedidos con nombre_producto = "Desconocido" del lado del costo (su costo_total_pedido es NaN y no se sumó), aunque su ingreso (monto_total) sí está incluido en el ingreso total — esto significa que el costo total real podría ser ligeramente mayor, y el profit ligeramente menor, de lo que refleja este número. Es un efecto mínimo (30 de 24,940 pedidos, 0.12%)

**Calculo de KPI'S**

In [ ]:
# Ticket promedio: monto_total promedio por orden.
ticket_promedio = orders_full["monto_total"].mean()

# Cantidad promedio de productos por orden.
cantidad_promedio = orders_full["cantidad"].mean()

print(f"Ticket promedio: ${ticket_promedio:,.2f}")
print(f"Cantidad promedio por orden: {cantidad_promedio:.2f}")

Ticket promedio: $385.77
Cantidad promedio por orden: 1.50


In [ ]:
# Producto más vendido por cantidad total (no por número de pedidos).
producto_mas_vendido = orders_full.groupby("nombre_producto")["cantidad"].sum().sort_values(ascending=False)
producto_mas_vendido

nombre_producto
Vacuum-Pro-Black        6284.0
Blender-XL-Red          6279.0
Jacket-Winter-M         6256.0
Sneakers-Urban-42       6172.0
Laptop-Gaming-16GB      4198.0
Tablet-Standard-64GB    4153.0
Phone-Pro-128GB         4135.0
Desconocido               45.0
Name: cantidad, dtype: float64

In [ ]:
# Gasto total en marketing por canal, directo de marketing_clean.
gasto_por_canal = marketing_clean.groupby("canal")["gasto"].sum().sort_values(ascending=False)
gasto_por_canal

canal
social         976818.37
organic        972650.96
paid_search    922374.20
Name: gasto, dtype: float64

**Comportamiento de ventas**

Producto más vendido es muy cerrado: Vacuum-Pro-Black (6,284) apenas supera a Blender-XL-Red (6,279) y Jacket-Winter-M (6,256) — una diferencia de menos del 0.5% entre los tres primeros lugares. No hay un producto claramente dominante; el catálogo está bastante parejo en volumen, salvo Laptop-Gaming-16GB (4,198) y Phone-Pro-128GB (4,135), que quedan visiblemente por debajo del resto.

Gasto en marketing muy balanceado entre canales: los tres canales están dentro de un rango de apenas $54,444 de diferencia (menos del 6% entre el más alto y el más bajo) — parece una estrategia de inversión deliberadamente distribuida, no concentrada en un canal ganador.

Cantidad promedio de 1.50 — la mayoría de los pedidos son de 1-2 unidades, coherente con lo que vimos en las estadísticas descriptivas originales (mediana de cantidad en 2.0).

---

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [ ]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [ ]:
# Explorar columna events
# =========================
query_events = '''
-- Explora los valores únicos de nombre_evento y cuántos registros
-- tiene cada uno, para entender el volumen relativo antes de definir
-- el orden lógico del funnel.
SELECT
    nombre_evento,
    COUNT(*) AS total_registros,
    COUNT(DISTINCT id_usuario) AS usuarios_unicos
FROM events
GROUP BY nombre_evento
ORDER BY total_registros DESC;
'''
events = pd.read_sql(query_events, con=engine)
events

,nombre_evento,total_registros,usuarios_unicos
0,first_visit,29957,7796
1,add_to_cart,24157,7634
2,select_item,23887,7582
3,begin_checkout,17971,7208
4,add_payment_info,12018,6250
5,purchase,12010,6240


Definí el orden según el flujo lógico de negocio (first_visit → select_item → add_to_cart → begin_checkout → add_payment_info → purchase), no por volumen.

Notese que add_to_cart (24,157) tiene más registros que select_item (23,887) — es decir, hay usuarios que llegan a "agregar al carrito" sin pasar por "seleccionar producto" registrado. Esto puede ser normal (ej. un botón de "agregar rápido" en un listado, sin abrir el detalle del producto) o una señal de que el evento select_item no se está registrando bien en algunos casos. Vale la pena tenerlo en mente al interpretar el funnel.

In [ ]:
# PARTE 1: Totales del funnel
# ======================

query_totals = '''
-- Define el orden lógico del funnel según el flujo real del usuario
-- en el negocio, no por volumen de eventos.
WITH orden_funnel AS (
    SELECT 'first_visit'       AS nombre_evento, 1 AS orden_etapa
    UNION ALL SELECT 'select_item',        2
    UNION ALL SELECT 'add_to_cart',        3
    UNION ALL SELECT 'begin_checkout',     4
    UNION ALL SELECT 'add_payment_info',   5
    UNION ALL SELECT 'purchase',           6
),

-- Cuenta usuarios ÚNICOS por etapa (no eventos totales), ya que un
-- mismo usuario puede generar el mismo evento varias veces
-- (ej. agregar varios productos al carrito) y debe contar una sola vez.
usuarios_por_etapa AS (
    SELECT
        nombre_evento,
        COUNT(DISTINCT id_usuario) AS usuarios_unicos
    FROM events
    GROUP BY nombre_evento
)

SELECT
    o.orden_etapa,
    o.nombre_evento,
    COALESCE(u.usuarios_unicos, 0) AS usuarios_unicos
FROM orden_funnel o
LEFT JOIN usuarios_por_etapa u
    ON o.nombre_evento = u.nombre_evento
ORDER BY o.orden_etapa;
'''

totals = pd.read_sql(query_totals, con=engine)
totals

,orden_etapa,nombre_evento,usuarios_unicos
0,1,first_visit,7796
1,2,select_item,7582
2,3,add_to_cart,7634
3,4,begin_checkout,7208
4,5,add_payment_info,6250
5,6,purchase,6240


In [ ]:
# PARTE 2: Conversiones
# ======================

query_conversion = '''
WITH orden_funnel AS (
    SELECT 'first_visit'       AS nombre_evento, 1 AS orden_etapa
    UNION ALL SELECT 'select_item',        2
    UNION ALL SELECT 'add_to_cart',        3
    UNION ALL SELECT 'begin_checkout',     4
    UNION ALL SELECT 'add_payment_info',   5
    UNION ALL SELECT 'purchase',           6
),

usuarios_por_etapa AS (
    SELECT
        nombre_evento,
        COUNT(DISTINCT id_usuario) AS usuarios_unicos
    FROM events
    GROUP BY nombre_evento
),

funnel_base AS (
    SELECT
        o.orden_etapa,
        o.nombre_evento,
        COALESCE(u.usuarios_unicos, 0) AS usuarios_unicos
    FROM orden_funnel o
    LEFT JOIN usuarios_por_etapa u
        ON o.nombre_evento = u.nombre_evento
),

-- LAG trae el conteo de la etapa anterior en la misma fila, para
-- calcular la conversión paso a paso sin un self-join.
funnel_con_anterior AS (
    SELECT
        orden_etapa,
        nombre_evento,
        usuarios_unicos,
        LAG(usuarios_unicos) OVER (ORDER BY orden_etapa) AS usuarios_etapa_anterior,
        FIRST_VALUE(usuarios_unicos) OVER (ORDER BY orden_etapa) AS usuarios_etapa_inicial
    FROM funnel_base
)

SELECT
    orden_etapa,
    nombre_evento,
    usuarios_unicos,
    -- Conversión respecto al paso inmediato anterior
    ROUND(
        100.0 * usuarios_unicos / NULLIF(usuarios_etapa_anterior, 0), 2
    ) AS conversion_paso_anterior_pct,
    -- Usuarios perdidos respecto al paso anterior (drop-off)
    usuarios_etapa_anterior - usuarios_unicos AS usuarios_perdidos,
    -- Conversión acumulada desde el inicio del funnel (first_visit)
    ROUND(
        100.0 * usuarios_unicos / NULLIF(usuarios_etapa_inicial, 0), 2
    ) AS conversion_acumulada_pct
FROM funnel_con_anterior
ORDER BY orden_etapa;
'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion

,orden_etapa,nombre_evento,usuarios_unicos,conversion_paso_anterior_pct,usuarios_perdidos,conversion_acumulada_pct
0,1,first_visit,7796,NaN,NaN,100.00
1,2,select_item,7582,97.26,214.0,97.26
2,3,add_to_cart,7634,100.69,-52.0,97.92
3,4,begin_checkout,7208,94.42,426.0,92.46
4,5,add_payment_info,6250,86.71,958.0,80.17
5,6,purchase,6240,99.84,10.0,80.04


**Hallazgos**

add_to_cart tiene un valor imposible:

usuarios_perdidos = -52.0 en la etapa add_to_cart, y conversion_paso_anterior_pct = 100.69% — una tasa de conversión mayor al 100% es matemáticamente imposible en un funnel estricto, y confirma lo que sospechaba desde el diagnóstico inicial: hay usuarios que llegan a add_to_cart (7,634) sin haber pasado por select_item (7,582) registrado. Esto no es un error de la query — es un problema de instrumentación/tracking en el evento select_item, que no se está capturando en todos los flujos donde el usuario sí agrega productos al carrito (ej. un botón de "agregar rápido" desde un listado, sin abrir el detalle del producto).

La calidad de los datos afecta el analisis en este caso.

1. ¿La retención/conversión cae drásticamente en el primer paso? (Problema de onboarding)
No — la caída inicial (first_visit → select_item) es de solo 2.74% (97.26% de conversión), la más baja de todo el funnel. El "onboarding" hacia la exploración de productos no es un problema.

2. ¿Dónde se pierde la mayor cantidad de usuarios?
begin_checkout → add_payment_info, sin ambigüedad — es la caída más grande tanto en % (86.71% de conversión, la más baja de todo el funnel) como en volumen absoluto (958 usuarios perdidos, más del doble que cualquier otra etapa). Este es el cuello de botella real: casi 1 de cada 7 usuarios que inician el checkout abandona antes de llegar a ingresar su método de pago. Vale la pena investigar aquí: ¿el formulario de pago es muy largo? ¿hay fricción con métodos de pago disponibles? ¿costos ocultos que se revelan en ese paso (envío, impuestos)?

3. ¿Se estabiliza en algún punto? (Usuarios comprometidos)
Sí — de add_payment_info a purchase la conversión sube a 99.84%, casi perfecta. Esto es una señal muy positiva: una vez que el usuario decide ingresar su información de pago, casi nadie abandona antes de completar la compra. El problema no está en "cerrar la venta", está exclusivamente en llegar a esa etapa de pago.

Tasa de conversión final

80.04% (first_visit → purchase) — alto para un funnel de e-commerce (el benchmark típico de la industria suele rondar 2-5% cuando se mide desde tráfico general, pero aquí el first_visit ya parece ser tráfico calificado/con intención, no visitas frías, así que este número tiene sentido en ese contexto).

---

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users`
- `user_activity`

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.
2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [ ]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [ ]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT *
FROM user_activity;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1


In [ ]:
# Retención por cohortes
# ======================
from sqlalchemy import text
query_cohort_retention_final = '''
-- 1. Identifica la cohorte de cada usuario, el mes de su fecha_registro.
-- Se castea fecha_registro a tipo date, ya que la columna esta
-- almacenada como texto y DATE_TRUNC requiere un tipo de fecha explicito.
WITH cohortes AS (
    SELECT
        id_usuario,
        fecha_registro,
        DATE_TRUNC('month', fecha_registro::date) AS mes_cohorte
    FROM users
),

actividad_con_semana AS (
    SELECT
        c.id_usuario,
        c.mes_cohorte,
        FLOOR(a.dias_despues_registro / 7) AS semanas_desde_registro
    FROM cohortes c
    JOIN user_activity a
        ON c.id_usuario = a.id_usuario
    WHERE a.activo = 1
),

tamano_cohorte AS (
    SELECT
        mes_cohorte,
        COUNT(DISTINCT id_usuario) AS total_usuarios
    FROM cohortes
    GROUP BY mes_cohorte
),

activos_por_semana AS (
    SELECT
        mes_cohorte,
        semanas_desde_registro,
        COUNT(DISTINCT id_usuario) AS usuarios_activos
    FROM actividad_con_semana
    GROUP BY mes_cohorte, semanas_desde_registro
)

SELECT
    a.mes_cohorte,
    a.semanas_desde_registro,
    a.usuarios_activos,
    t.total_usuarios,
    ROUND(100.0 * a.usuarios_activos / t.total_usuarios, 2) AS tasa_retencion_pct
FROM activos_por_semana a
JOIN tamano_cohorte t
    ON a.mes_cohorte = t.mes_cohorte
ORDER BY a.mes_cohorte, a.semanas_desde_registro;
'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(text(query_cohort_retention_final), con=engine)
cohorte_final

,mes_cohorte,semanas_desde_registro,usuarios_activos,total_usuarios,tasa_retencion_pct
0,2025-01-01 00:00:00+00:00,1.0,697,1627,42.84
1,2025-01-01 00:00:00+00:00,2.0,668,1627,41.06
2,2025-01-01 00:00:00+00:00,3.0,656,1627,40.32
3,2025-01-01 00:00:00+00:00,4.0,671,1627,41.24
4,2025-02-01 00:00:00+00:00,1.0,611,1444,42.31
5,2025-02-01 00:00:00+00:00,2.0,609,1444,42.17
6,2025-02-01 00:00:00+00:00,3.0,635,1444,43.98
7,2025-02-01 00:00:00+00:00,4.0,575,1444,39.82
8,2025-03-01 00:00:00+00:00,1.0,677,1636,41.38
9,2025-03-01 00:00:00+00:00,2.0,705,1636,43.09


**Hallazgos**

1. ¿La retención cae drásticamente después de la semana 1? (Problema de onboarding)

No. Comparando la semana 1 contra las semanas 2, 3 y 4 en cada cohorte, la retención se mantiene dentro de una banda muy angosta (~39.8% a ~44%), sin ninguna caída pronunciada ni tendencia sostenida hacia abajo:

La variación semana a semana se comporta más como ruido normal que como una señal de abandono progresivo. No hay evidencia de un problema de onboarding en esta ventana.

2. ¿Se estabiliza en algún punto? (Usuarios comprometidos)

Sí, desde la primera semana observada. La retención se estabiliza rápido y se sostiene alrededor del 40-44% durante las 4 semanas analizadas en todas las cohortes, sin degradación progresiva. Esto sugiere una base de usuarios recurrentes consistente una vez que superan la primera semana de actividad.

3. ¿Hay cohortes que retienen mejor que otras? (¿Qué cambió?)

No hay diferencias significativas entre cohortes. Las 5 cohortes (enero a mayo 2025) oscilan en el mismo rango general de retención, sin que ninguna destaque claramente por encima o por debajo del resto. Esto indica que no hubo ningún cambio de producto, canal de adquisición o campaña que haya alterado sustancialmente el comportamiento de retención mes a mes — el patrón es consistente en el tiempo.

Conclusión general

La retención del producto es estable pero moderada (~40-44%), sin señales de deterioro por onboarding ni diferencias relevantes entre cohortes. Esto sugiere que el problema, si existe, no está en qué tan bien se retiene a los usuarios que ya llegaron a la primera semana, sino potencialmente en la conversión previa (de registro a esa primera semana activa)

---

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado**
4. **Interpretar el resultado**  

---
Hipótesis estadística
   - **H₀ (Hipótesis nula):** ...
   - **H₁ (Hipótesis alternativa):** ...
   
**Test estadístico:** ...  
**Nivel de significancia alpha:** ...

---

**Analisis exploratorio inicial de los datos**

In [ ]:
experiment = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv')
display(experiment.head())
experiment.info()

,id_usuario,variante,convirtio,dispositivo,pais,duracion_sesion,timestamp
0,exp_user_0,tratamiento,0,mobile,Argentina,114.41,2025-03-28
1,exp_user_1,tratamiento,0,desktop,Mexico,170.03,2025-01-15
2,exp_user_2,control,1,mobile,Colombia,140.21,2025-03-18
3,exp_user_3,tratamiento,0,mobile,Colombia,151.45,2025-06-03
4,exp_user_4,tratamiento,0,desktop,Mexico,299.96,2025-01-12


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id_usuario       10000 non-null  object 
 1   variante         10000 non-null  object 
 2   convirtio        10000 non-null  int64  
 3   dispositivo      10000 non-null  object 
 4   pais             10000 non-null  object 
 5   duracion_sesion  10000 non-null  float64
 6   timestamp        10000 non-null  object 
dtypes: float64(1), int64(1), object(5)
memory usage: 547.0+ KB


**Analisis de la distribucion de usuarios**

In [ ]:
# Explora la distribución de usuarios entre variantes (control vs tratamiento)
# y la tasa de conversión de cada una — esta es la comparación central  de todo el análisis.
resumen_variantes = experiment.groupby("variante").agg(
    usuarios=("id_usuario", "count"),
    conversiones=("convirtio", "sum"),
    tasa_conversion=("convirtio", "mean")
)
resumen_variantes["tasa_conversion_pct"] = (resumen_variantes["tasa_conversion"] * 100).round(2)
resumen_variantes

,usuarios,conversiones,tasa_conversion,tasa_conversion_pct
variante,,,,
control,4965,779,0.156898,15.69
tratamiento,5035,820,0.162860,16.29


Hay un buen balance de muestra (4,965 vs 5,035, prácticamente 50/50) y las tasas de conversión son cercanas: control 15.69% vs tratamiento 16.29% — una diferencia de apenas 0.60 puntos porcentuales. Con una diferencia tan pequeña, mi expectativa es que el test probablemente no va a resultar significativo

**Z de proporciones por los tipos de dato**

In [ ]:
from statsmodels.stats.proportion import proportions_ztest

# Conteos de conversión y tamaño de muestra por grupo, en el orden
# [tratamiento, control] para que el signo del resultado sea intuitivo
# (positivo = tratamiento convierte más que control).
conversiones = [820, 779]
n_usuarios = [5035, 4965]

# Ejecuta el test de dos proporciones (z-test), de dos colas
z_stat, p_valor = proportions_ztest(conversiones, n_usuarios, alternative='two-sided')

print(f"Estadístico Z: {z_stat:.4f}")
print(f"P-valor: {p_valor:.4f}")

Estadístico Z: 0.8133
P-valor: 0.4161


In [ ]:
alpha = 0.05

if p_valor < alpha:
    print(f"P-valor ({p_valor:.4f}) < alpha ({alpha}) → Se rechaza H0.")
    print("Hay evidencia estadísticamente significativa de que la modificación")
    print("en la UI del checkout SÍ impacta la tasa de conversión.")
else:
    print(f"P-valor ({p_valor:.4f}) >= alpha ({alpha}) → No se rechaza H0(hipotesis nula).")
    print("No hay evidencia suficiente para afirmar que la modificación en la")
    print("UI del checkout impacta la tasa de conversión de forma significativa.")

# Diferencia absoluta en puntos porcentuales, para contextualizar
# la magnitud del efecto más allá de la significancia estadística.
diferencia_pp = (820/5035 - 779/4965) * 100
print(f"\nDiferencia absoluta: {diferencia_pp:.2f} puntos porcentuales")

P-valor (0.4161) >= alpha (0.05) → No se rechaza H0(hipotesis nula).
No hay evidencia suficiente para afirmar que la modificación en la
UI del checkout impacta la tasa de conversión de forma significativa.

Diferencia absoluta: 0.60 puntos porcentuales


---

El experimento no encontró un efecto detectable. No se puede afirmar que el nuevo diseño de checkout mejore (ni empeore) la conversión respecto al diseño actual.

"No significativo" no es lo mismo que "no hay diferencia real" — puede haber un efecto verdadero pero pequeño que este experimento no tuvo el tamaño de muestra suficiente para detectar (poder estadístico insuficiente). Con ~10,000 usuarios totales, el test puede no ser lo bastante sensible para diferencias de menos de 1 punto porcentual. Si el equipo de producto considera que incluso una mejora de 0.6 puntos porcentuales sería valiosa a escala, valdría la pena correr el experimento con una muestra mayor antes de descartar el cambio por completo.

Decisión recomendada: no hay justificación estadística para lanzar el nuevo diseño de checkout basándose en conversión únicamente. Si el equipo tiene otras razones para el cambio (ej. mejoras de accesibilidad, reducción de errores de soporte, preferencia cualitativa de usuarios), esas razones tendrían que sostener la decisión por separado, no la conversión.

---

## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

---